# 05 - Topic Greenlist Construction

Builds one greenlist per topic (8 topics) from the
`topic_cosine_similarity.csv` produced by notebook 04.

1. **Segregate connector words** (topic-neutral function words like *is, a,
   then*) into a separate list.
2. **Threshold on cosine similarity** (token vs. topic). A token joins the
   greenlist of the topic it is *most* similar to, only if that similarity
   `>= THRESHOLD`. Everything else goes to a redlist.
3. Connector words are kept apart so they can later be **round-robined** into
   each topic's greenlist (cell at the end).

> Note: the paper's detection-time threshold (τ=0.7) applies to a z-score /
> false-positive scale, *not* to token↔topic cosine similarity. On our scale
> token similarities peak around 0.25, so `THRESHOLD` here is a much smaller
> number (default 0.15).

In [10]:
import os
from collections import defaultdict

import numpy as np
import pandas as pd


DATA_DIR = "/content/drive/MyDrive/minor_project/"
for cand in ("../data/complete/", "data/complete/"):
    if not os.path.isdir(DATA_DIR) and os.path.isfile(
            os.path.join(cand, "topic_cosine_similarity.csv")):
        DATA_DIR = cand
if not os.path.isfile(os.path.join(DATA_DIR, "topic_cosine_similarity.csv")):
    raise FileNotFoundError(f"topic_cosine_similarity.csv not found under {DATA_DIR}")

THRESHOLD = 0.15  # min cosine similarity (token vs topic) to join a greenlist

# One greenlist per topic; philosophy is dropped (8 greenlists total).
TOPICS = [
    "technology",
    "medicine",
    "sports",
    "politics",
    "science",
    "entertainment",
    "finance",
    "history",
]
sim_cols = [f"sim_{t}" for t in TOPICS]
print("topics:", TOPICS)

topics: ['technology', 'medicine', 'sports', 'politics', 'science', 'entertainment', 'finance', 'history']


In [11]:
# ------------------------------------------------------------------
# Load the topic-cosine-similarity table
# ------------------------------------------------------------------
df = pd.read_csv(os.path.join(DATA_DIR, "topic_cosine_similarity.csv")).dropna(subset=["token"])

# OPT BPE tokens carry a "Ġ" space-marker; strip it and lowercase for matching.
df["clean"] = df["token"].str.replace("Ġ", "", regex=False).str.lower()

print("rows:", len(df))
print("columns:", list(df.columns))

rows: 50260
columns: ['token_id', 'token', 'sim_technology', 'sim_medicine', 'sim_sports', 'sim_politics', 'sim_science', 'sim_entertainment', 'sim_finance', 'sim_history', 'clean']


In [12]:
# ------------------------------------------------------------------
# 1) Segregate connector words
# ------------------------------------------------------------------
# Topic-neutral function words: articles, prepositions, conjunctions,
# pronouns, auxiliaries and common transitional adverbs. They rarely clear
# the cosine threshold for any single topic, so we pull them out now and will
# round-robin them across the topic greenlists later.
CONNECTOR_WORDS = {
    "a", "an", "the",
    "and", "or", "but", "nor", "so", "yet", "for",
    "of", "in", "on", "at", "to", "from", "by", "with", "about",
    "above", "across", "after", "against", "along", "among", "around", "as",
    "before", "behind", "below", "beneath", "beside", "between", "beyond",
    "during", "except", "inside", "into", "near", "off", "onto", "out",
    "outside", "over", "past", "through", "throughout", "till", "toward",
    "under", "until", "up", "upon", "within", "without",
    "is", "are", "was", "were", "be", "been", "being", "am",
    "do", "does", "did", "have", "has", "had",
    "will", "would", "shall", "should", "can", "could", "may", "might", "must",
    "i", "you", "he", "she", "it", "we", "they", "me", "him", "her", "us", "them",
    "my", "your", "his", "its", "our", "their", "mine", "yours", "hers", "ours", "theirs",
    "this", "that", "these", "those", "who", "whom", "whose", "which", "what",
    "whatever", "whoever",
    "then", "than", "if", "unless", "while", "when", "where", "whether", "how", "why",
    "although", "though", "because",
    "not", "no", "only", "just", "very", "also", "too", "such", "here", "there", "now",
    "however", "therefore", "moreover", "furthermore", "nevertheless", "meanwhile",
    "finally", "first", "second", "next", "last", "again",
    "already", "always", "ever", "never", "often", "sometimes", "usually",
    "still", "quite", "rather", "almost", "nearly", "even", "well",
}

conn_mask = df["clean"].isin(CONNECTOR_WORDS)
connectors = df.loc[conn_mask, ["token_id", "token", "clean"]].copy()
main_df = df.loc[~conn_mask].copy()

# Drop special tokens (<s>, </s>, <pad>, <unk>) — they are not real words and
# would pollute the greenlists.
SPECIAL_TOKENS = {"<s>", "</s>", "<pad>", "<unk>"}
main_df = main_df[~main_df["clean"].isin(SPECIAL_TOKENS)].copy()

print(f"connector words:    {len(connectors):>7}  ({100 * len(connectors) / len(df):.2f}% of vocab)")
print(f"non-connector vocab: {len(main_df):>7}  ({100 * len(main_df) / len(df):.2f}% of vocab)")
print("\nsample connectors:")
print(connectors["token"].head(15).tolist())

connector words:        735  (1.46% of vocab)
non-connector vocab:   49521  (98.53% of vocab)

sample connectors:
['Ġthe', 'Ġto', 'Ġand', 'Ġof', 'Ġa', 'Ġin', 'Ġfor', 'Ġthat', 'Ġon', 'Ġis', 'Ġwith', 'ĠThe', 'Ġwas', 'Ġat', 'Ġit']


In [13]:

best_col = main_df[sim_cols].idxmax(axis=1)
best_sim = main_df[sim_cols].max(axis=1)

main_df["topic"] = best_col.str.replace("sim_", "", regex=False)
main_df["similarity"] = best_sim.round(4)
main_df["greenlist"] = best_sim >= THRESHOLD

print("best-topic assignment counts (all non-connectors):")
print(main_df["topic"].value_counts().to_string())

best-topic assignment counts (all non-connectors):
topic
technology       14313
politics          9826
sports            7465
science           5842
history           5596
entertainment     2598
medicine          2173
finance           1708


In [14]:
# ------------------------------------------------------------------
# 3) Build the per-topic greenlists
# ------------------------------------------------------------------
greenlists = {}
for t in TOPICS:
    g = main_df[(main_df["topic"] == t) & main_df["greenlist"]]
    greenlists[t] = g["token_id"].tolist()

redlist = main_df[~main_df["greenlist"]]

print(f"threshold: {THRESHOLD}")
print(f"greenlisted tokens: {sum(len(v) for v in greenlists.values()):,}  ({100 * main_df['greenlist'].mean():.1f}% of non-connector vocab)")
print(f"redlist tokens:     {len(redlist):,}")
print("\nper-topic greenlist sizes:")
for t in TOPICS:
    print(f"  {t:15s} {len(greenlists[t]):>7}")

threshold: 0.15
greenlisted tokens: 17,181  (34.7% of non-connector vocab)
redlist tokens:     32,340

per-topic greenlist sizes:
  technology         7105
  medicine             95
  sports             2575
  politics           3566
  science            1958
  entertainment       110
  finance             102
  history            1670


In [15]:
# ------------------------------------------------------------------
# Preview: top tokens in a couple of greenlists
# ------------------------------------------------------------------
for t in ["technology", "sports"]:
    g = main_df[(main_df["topic"] == t) & main_df["greenlist"]]
    print(f"\n=== greenlist '{t}' (top 20 by similarity) ===")
    print(g.sort_values("similarity", ascending=False)[["token", "similarity"]].head(20).to_string(index=False))


=== greenlist 'technology' (top 20 by similarity) ===
           token  similarity
      technology      1.0000
      Technology      0.7134
     ĠTechnology      0.6306
     Ġtechnology      0.6282
   Ġtechnologies      0.5620
  Ġtechnological      0.5091
           ĠTECH      0.5017
Ġtechnologically      0.4609
            tech      0.4376
     otechnology      0.4127
           Ġtech      0.4063
           techn      0.3922
            Tech      0.3909
   ĠTechnologies      0.3786
       technical      0.3751
         devices      0.3544
           ĠTech      0.3534
        chnology      0.3485
        software      0.3466
         Ġtechno      0.3422

=== greenlist 'sports' (top 20 by similarity) ===
     token  similarity
    sports      1.0000
    Sports      0.6215
   Ġsports      0.5811
   ĠSPORTS      0.5359
   ĠSports      0.5314
     Sport      0.4299
 Ġsporting      0.3919
Ġathletics      0.3774
    Ġsport      0.3736
    ĠSport      0.3725
basketball      0.3530
 Ġathlete

In [16]:
# ------------------------------------------------------------------
# 4) Persist greenlists + the segregated connector list
# ------------------------------------------------------------------
out_dir = os.path.join(DATA_DIR)
os.makedirs(out_dir, exist_ok=True)

# Long-form table: one row per greenlisted token
rows = []
for t in TOPICS:
    g = main_df[(main_df["topic"] == t) & main_df["greenlist"]]
    rows.append(pd.DataFrame({
        "topic": t,
        "token_id": g["token_id"],
        "token": g["token"],
        "similarity": g["similarity"],
    }))
green_df = pd.concat(rows, ignore_index=True)
green_df.to_csv(os.path.join(out_dir, "topic_greenlists.csv"), index=False)

# Segregated connector words, ready for round-robin distribution
connectors.to_csv(os.path.join(out_dir, "connector_words.csv"), index=False)

print(f"saved topic_greenlists.csv  ({len(green_df):,} rows)")
print(f"saved connector_words.csv   ({len(connectors):,} rows)")
print(green_df.head(5).to_string(index=False))

saved topic_greenlists.csv  (17,181 rows)
saved connector_words.csv   (735 rows)
     topic  token_id       token  similarity
technology       806 Ġtechnology      0.6282
technology      2903       Ġtech      0.4063
technology      3165  Ġtechnical      0.1689
technology      3777 ĠTechnology      0.6306
technology      3815      ĠNIGHT      0.1886


In [17]:
# Distributes the segregated connector tokens evenly across the 8 greenlists
# so that topic-greenlist keys are available even when the model emits a
# connector word mid-sentence.
def round_robin_connectors(greenlists, connector_ids):
    # Return a copy of greenlists with connectors distributed evenly
    # across the topic greenlists.
    topics = list(greenlists)
    out = {t: list(ids) for t, ids in greenlists.items()}
    for i, tid in enumerate(connector_ids):
        out[topics[i % len(topics)]].append(tid)
    return out

# Example: with_connectors = round_robin_connectors(greenlists, connectors["token_id"].tolist())
print("round_robin_connectors() defined — call it when you want to fold",
      "connector words into the per-topic greenlists.")

round_robin_connectors() defined — call it when you want to fold connector words into the per-topic greenlists.


In [18]:
round_robin_greenlists = round_robin_connectors(greenlists, connectors["token_id"].tolist())